# 02 – The Intern: Fine-Tuning (Baseline)

**Operation Ledger-Mind: The Financial Intelligence**

**Model:** `Llama 3.2 3B Instruct` | **Precision:** `FP16` | **Strategy:** `LoRA (No Quantization)`

> This notebook uses the standard HuggingFace fine-tuning stack without quantization.
> Runs on T4 (Colab free tier) using the 3B model which fits in 16GB VRAM natively.
> To run the 8B model with 4-bit QLoRA on T4, use `02_finetuning_intern_qlora.ipynb` instead.
> Uncomment `quantization_config=bnb_config` and switch to an 8B model if running on a local Ampere+ GPU (RTX 3090, A100, etc.) that supports BF16.

In [ ]:
# Setup
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
import json
import torch

!pip install --upgrade torchao peft
!pip install -q trl peft accelerate transformers
!pip install -U bitsandbytes

sys.path.append(str(Path.cwd().parent))
load_dotenv()

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6GB


In [ ]:
# Load training data
train_file = "/content/train.jsonl"

train_data = []
with open(train_file) as f:
    for line in f:
        train_data.append(json.loads(line))

print(f"Loaded {len(train_data)} training examples")

Loaded 1154 training examples


In [ ]:
# Format for instruction tuning
from datasets import Dataset

def format_instruction(example):
    return {
        "messages": [
            {"role": "system", "content": "You are a financial analyst trained on Uber's 2024 Annual Report."},
            {"role": "user", "content": example['question']},
            {"role": "assistant", "content": example['answer']}
        ]
    }

formatted = [format_instruction(ex) for ex in train_data]
dataset = Dataset.from_list(formatted)
print(f"Formatted {len(dataset)} examples")

Formatted 1154 examples


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

model_id = "meta-llama/Llama-3.2-3B-Instruct"  # or 8B

# use if cuda available
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# use if cuda available
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    # quantization_config=bnb_config,               # uncomment to switch to 4-bit QLoRA
    dtype=torch.float16,
    attn_implementation="eager"
)

model.config.dtype = torch.float16

print("Model loaded in 16-bit")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Model loaded in 16-bit


In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} ({100*trainable/total:.2f}%)")

Trainable: 9,175,040 (0.28%)


In [ ]:
# Training
output_dir = "/content/finetuned_model/"

training_args = SFTConfig(
    output_dir=str(output_dir),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_steps=50,
    fp16=True,
    bf16=False,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

print("Training...")
trainer.train()
print("Training complete!")

Tokenizing train dataset:   0%|          | 0/1154 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Training...


Step,Training Loss
10,3.693896
20,1.545643
30,0.992992
40,1.050407
50,1.031048
60,0.968134
70,0.954270
80,0.970823
90,0.892490
100,1.115472


Training complete!


In [ ]:
# Save adapters and create inference function
adapter_path = "/content/finetuned_model/final_adapters_baseline"
model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))

def query_intern(question, max_tokens=256):
    model.eval()

    messages = [
        {"role": "system", "content": "Financial analyst for Uber 2024."},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.1,
            do_sample=True,
            use_cache=True
        )

    # ONLY decode new tokens
    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)

    return response.strip()

print("query_intern() function ready")
print("\nTesting...")
print(query_intern("What is Uber's mission?"))
print("\nThe Intern is ready!")

query_intern() function ready

Testing...
To create a world where everyone can move freely and easily

The Intern is ready!
